# Evaluation and results

The third of three notebooks. The first covers the data pipeline and the metric, the
second the models and how they are trained, this one the evaluation and the figures used
in the report.

Reads the CSV files produced by `scripts/evaluate.py`. No model is trained here, and no
evaluation is run: this notebook reads, aggregates and plots.

`RUNS` must point at a directory holding one subdirectory per run, each containing
`eval_test.csv` and `eval_val.csv`. Point it at the downloaded archive of the Kaggle runs,
or at a local `runs/` if training was done locally.

Confidence intervals are standard deviations over training seeds, that is over
independently trained models.

In [ ]:
import csv
import os
import sys

sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import numpy as np

RUNS = "../runs"
SPLIT = "test"           # ablations were selected on validation; final numbers on test
FIGDIR = "../figures"
os.makedirs(FIGDIR, exist_ok=True)

plt.rcParams.update({"figure.dpi": 130, "font.size": 9,
                     "axes.grid": True, "grid.alpha": 0.3})

M2      = ["m2_seed1"] + [f"m2_s{s}" for s in range(2, 9)]
M1      = [f"m1_null_s{s}" for s in (1, 2, 3)]
UNET_1  = [f"unet_oneshot_s{s}" for s in (1, 2, 3)]
UNET_16 = [f"unet_iter16_s{s}" for s in (1, 2, 3)]
AUX     = [f"aux10_s{s}" for s in range(1, 9)]
HIDDEN  = {h: [f"h{h}_s{s}" for s in (1, 2, 3)] for h in (8, 16, 24)}
BPTT    = {b: [f"bp{b}_s{s}" for s in (1, 2, 3)] for b in (64, 128)}
LAP     = [f"lap_s{s}" for s in range(1, 9)]

In [ ]:
def read_run(run, split=SPLIT):
    path = os.path.join(RUNS, run, f"eval_{split}.csv")
    if not os.path.exists(path):
        return {}
    with open(path) as f:
        return {row["damage"]: row for row in csv.DictReader(f)}


def stat(runs, condition, field="rsr_mean", split=SPLIT):
    vals = [float(d[condition][field])
            for d in (read_run(r, split) for r in runs)
            if condition in d]
    return (np.mean(vals), np.std(vals), len(vals)) if vals else (np.nan, np.nan, 0)


def table(rows, conditions, title, split=SPLIT):
    print(f"\n{title}")
    print(f"{'model':<28}" + "".join(f"{c:>16}" for c in conditions))
    print("-" * (28 + 16 * len(conditions)))
    for label, runs in rows:
        line = f"{label:<28}"
        n = 0
        for c in conditions:
            m, s, n = stat(runs, c, split=split)
            line += f"{m:>8.3f}+-{s:<6.3f}" if n else f"{'-':>16}"
        print(line + f"   (n={n})")


COND = ["A0_none", "A3_matched_random", "B1_door", "B3_isolation"]

available = [d for d in sorted(os.listdir(RUNS))
             if os.path.exists(os.path.join(RUNS, d, f"eval_{SPLIT}.csv"))] if os.path.isdir(RUNS) else []
print(f"runs with eval_{SPLIT}.csv: {len(available)}")

## Stability ceiling

An undamaged room evolved for the full number of repair steps is not restored perfectly:
the stochastic update perturbs the state at every step. The A0 condition measures this,
and its value is the ceiling for every other number. A result of 0.73 against a ceiling
of 0.97 is three quarters of what the model can reach, not of a perfect score.

In [ ]:
ceiling, ceiling_std, _ = stat(M2, "A0_none")
print(f"A0_none for the main model: {ceiling:.3f} +- {ceiling_std:.3f}")

## Damage during training induces regeneration

M1 is identical to M2 except that it never sees a damaged room. Its only task is
persistence, and it learns it well: it preserves as well as M2 and repairs far worse.

In [ ]:
table([("M1 (no damage)", M1), ("M2 (stochastic damage)", M2)],
      COND, "Table 1")

## Matched-extent dissociation

The three conditions destroy the same number of cells. The only difference is which
cells, and the outcome varies by close to an order of magnitude. The ordering follows
how far local plausibility agrees with the global requirement: floor surrounded by floor
is both locally and globally correct, whereas a door cell is surrounded by walls, so the
locally plausible completion is exactly the wrong one.

In [ ]:
targets = [("B3_isolation", "walkable cells next to a door"),
           ("A3_matched_random", "random walkable cells"),
           ("B1_door", "door cells")]

print(f"{'condition':<22}{'target of the K cells':<34}{'RSR':>16}{'% of ceiling':>14}")
print("-" * 86)
means = []
for cond, desc in targets:
    m, s, n = stat(M2, cond)
    means.append((m, s))
    print(f"{cond:<22}{desc:<34}{m:>9.3f}+-{s:<6.3f}{100 * m / ceiling:>12.0f}%")
print(f"\nratio B3/B1 = {means[0][0] / means[2][0]:.1f}x   (n={n} training seeds)")

fig, ax = plt.subplots(figsize=(5.2, 2.8))
labels = ["B3\nfloor next\nto a door", "A3\nrandom\nwalkable", "B1\ndoor\ncells"]
vals = [m for m, _ in means]
errs = [s for _, s in means]
bars = ax.bar(labels, vals, yerr=errs, capsize=4, width=0.55,
              color=["#3b7dd8", "#7aa6dd", "#d95f4c"])
ax.axhline(ceiling, ls=":", c="gray", lw=1.2)
ax.text(2.42, ceiling + 0.015, "ceiling (A0)", fontsize=7, color="gray", ha="right")
ax.set_ylabel("regeneration success rate")
ax.set_ylim(0, min(1.05, ceiling + 0.12))
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width() / 2, v + max(errs) + 0.03, f"{v:.2f}",
            ha="center", fontsize=8)
fig.savefig(f"{FIGDIR}/dissociation.png", bbox_inches="tight")
plt.show()

## Per-tile accuracy against topological success

The per-tile metric barely registers the failure that matters. This is the observation
that motivates a pathfinding-based criterion.

In [ ]:
print(f"{'condition':<22}{'tile accuracy':>16}{'RSR':>12}")
print("-" * 50)
for cond in ["A0_none", "B3_isolation", "A3_matched_random", "B1_door"]:
    acc, _, _ = stat(M2, cond, field="acc_mean")
    rsr, _, _ = stat(M2, cond)
    print(f"{cond:<22}{acc:>16.3f}{rsr:>12.3f}")

## Degradation under stochastic damage

Eight damage extents rather than three. The finer sampling costs no training, since it
reuses the trained checkpoints, and it locates the collapse instead of bracketing it.

In [ ]:
def curve(runs, damage="A1_erasure", split=SPLIT):
    by_extent = {}
    for r in runs:
        path = os.path.join(RUNS, r, f"eval_{split}.csv")
        if not os.path.exists(path):
            continue
        with open(path) as f:
            for row in csv.DictReader(f):
                if row["damage"] == damage:
                    by_extent.setdefault(float(row["extent"]), []).append(float(row["rsr_mean"]))
    x = sorted(by_extent)
    return x, np.array([np.mean(by_extent[k]) for k in x]), np.array([np.std(by_extent[k]) for k in x])


fig, ax = plt.subplots(figsize=(5.4, 3))
for runs, label, marker in [(M2, "erasure (A1)", "o")]:
    x, m, s = curve(runs)
    ax.errorbar(x, m, yerr=s, marker=marker, capsize=3, lw=1.6, ms=4, label=label)
x2, m2c, s2 = curve(M2, damage="A2_tileflip")
if len(x2):
    ax.errorbar(x2, m2c, yerr=s2, marker="s", capsize=3, lw=1.6, ms=4, ls="--",
                label="tile flip (A2)")
ax.set_xlabel("fraction of the room destroyed")
ax.set_ylabel("regeneration success rate")
ax.legend(frameon=False)
fig.savefig(f"{FIGDIR}/degradation.png", bbox_inches="tight")
plt.show()

x, m, _ = curve(M2)
print("  ".join(f"{a:.2f}->{b:.3f}" for a, b in zip(x, m)))

## Capacity does not move door repair

Hidden channels are the NCA's only means of carrying information across the grid.
Tripling them improves stability and the locally-consistent damage types, and leaves
door repair flat. This is the first of the alternative explanations to be ruled out.

In [ ]:
rows = [("hidden = 8", HIDDEN[8]), ("hidden = 12 (base)", M2),
        ("hidden = 16", HIDDEN[16]), ("hidden = 24", HIDDEN[24])]
table(rows, COND, "Capacity ablation", split="val")

fig, ax = plt.subplots(figsize=(5.4, 2.9))
hs = [8, 12, 16, 24]
for cond, marker in [("A0_none", "o"), ("B3_isolation", "^"),
                     ("A3_matched_random", "s"), ("B1_door", "D")]:
    ys = [stat(HIDDEN[h] if h != 12 else M2, cond, split="val")[0] for h in hs]
    ax.plot(hs, ys, marker=marker, lw=1.5, ms=4, label=cond)
ax.set_xticks(hs)
ax.set_xlabel("hidden channels")
ax.set_ylabel("regeneration success rate")
ax.set_ylim(0, 1.05)
ax.legend(frameon=False, fontsize=7, ncol=2)
fig.savefig(f"{FIGDIR}/capacity.png", bbox_inches="tight")
plt.show()

## Neither does a longer propagation horizon

A cell only talks to its neighbours, so information needs many steps to cross the room.
Doubling the unrolled horizon does not help; the highest value comes from the shortest
horizon. Every additional step re-applies the locally plausible completion, which erodes
a freshly restored door rather than consolidating it.

In [ ]:
table([("bptt_max = 64", BPTT[64]), ("bptt_max = 96 (base)", M2),
        ("bptt_max = 128", BPTT[128])],
      COND, "Temporal horizon ablation", split="val")

## Perception, and a note on low-seed variance

Adding a Laplacian to the fixed perception filters is the one architectural change that
appeared to move door repair. At three seeds the effect looked like a doubling; at eight
paired seeds it disappears, with a spread larger than the mean, and the other damage
types degrade. The three initial seeds were fortunate.

The comparison uses the same seeds on both arms: comparing different seeds across arms
would confound the filter with the initialisation.

In [ ]:
for n_seeds in (3, 8):
    sobel = M2[:n_seeds]
    lap = LAP[:n_seeds]
    ms, ss, ns = stat(sobel, "B1_door", split="val")
    ml, sl, nl = stat(lap, "B1_door", split="val")
    print(f"n={n_seeds}:  Sobel {ms:.3f}+-{ss:.3f} (n={ns})   "
          f"Laplacian {ml:.3f}+-{sl:.3f} (n={nl})")

table([("Sobel (base)", M2), ("with Laplacian", LAP)], COND,
      "Perception, eight paired seeds", split="val")

## Locality against a global view

With capacity, horizon and perception ruled out, the remaining candidate is the locality
of the rule itself. The U-Net is identical in interface, loss, pool and metric; only the
receptive field differs. It is used in two regimes because the NCA is both local and
iterative: comparing against a one-shot global model alone would move two variables at
once.

The outcome is a double dissociation. On the locally-consistent damage types the
iterative models lead and the one-shot model is worst; on door repair the ordering
reverses. Neither architecture dominates.

In [ ]:
table([("NCA (local, 96 steps)", M2),
        ("U-Net iterative-16", UNET_16),
        ("U-Net one-shot", UNET_1)],
      COND, "Table 2")

fig, ax = plt.subplots(figsize=(5.6, 2.9))
groups = ["A3_matched_random", "B3_isolation", "B1_door"]
idx = np.arange(len(groups))
width = 0.26
for k, (runs, label) in enumerate([(M2, "NCA (local, 96 steps)"),
                                   (UNET_16, "U-Net iterative-16"),
                                   (UNET_1, "U-Net one-shot")]):
    vals = [stat(runs, c)[0] for c in groups]
    errs = [stat(runs, c)[1] for c in groups]
    ax.bar(idx + (k - 1) * width, vals, width, yerr=errs, capsize=3, label=label)
ax.set_xticks(idx)
ax.set_xticklabels(["A3 random", "B3 isolation", "B1 door"])
ax.set_ylabel("regeneration success rate")
ax.set_ylim(0, 1.05)
ax.legend(frameon=False, fontsize=7)
fig.savefig(f"{FIGDIR}/local_vs_global.png", bbox_inches="tight")
plt.show()

## Making the topological signal explicit

One hidden channel is supervised towards the geodesic distance to the nearest access
point, computed on the pristine room. The architecture and the parameter count are
unchanged: the loss term is the only difference.

Door repair improves but does not close the gap with the global model, so the limitation
of locality is largely structural rather than a matter of what the rule can be taught.
The dose-response is informative on its own: past a certain weight the auxiliary task
takes over and reconstruction collapses.

In [ ]:
LAMBDAS = [("0", M2), ("0.1", [f"aux01_s{s}" for s in (1, 2, 3)]),
           ("1.0", AUX), ("10.0", [f"aux100_s{s}" for s in (1, 2, 3)])]

table([(f"lambda = {lab}", runs) for lab, runs in LAMBDAS], COND,
      "Table 3", split="val")

fig, ax = plt.subplots(figsize=(5.4, 2.9))
xs = np.arange(len(LAMBDAS))
b1 = [stat(r, "B1_door", split="val")[0] for _, r in LAMBDAS]
e1 = [stat(r, "B1_door", split="val")[1] for _, r in LAMBDAS]
a0 = [stat(r, "A0_none", split="val")[0] for _, r in LAMBDAS]
ax.errorbar(xs, b1, yerr=e1, marker="o", capsize=3, lw=1.6, ms=4, label="B1 door")
ax.plot(xs, a0, marker="s", ls="--", lw=1.4, ms=4, c="#888", label="A0 stability")
unet_b1, _, _ = stat(UNET_1, "B1_door", split="val")
ax.axhline(unet_b1, ls=":", c="#d95f4c", lw=1.3)
ax.text(xs[-1], unet_b1 + 0.02, "U-Net one-shot", fontsize=7, color="#d95f4c", ha="right")
ax.set_xticks(xs)
ax.set_xticklabels([lab for lab, _ in LAMBDAS])
ax.set_xlabel("weight of the auxiliary topological task")
ax.set_ylabel("regeneration success rate")
ax.set_ylim(0, 1.05)
ax.legend(frameon=False, fontsize=7)
fig.savefig(f"{FIGDIR}/multitask.png", bbox_inches="tight")
plt.show()

## Full per-condition results

The complete table for the appendix: every damage condition, both metrics.

In [ ]:
ALL_COND = ["A0_none", "A1_erasure", "A2_tileflip", "A3_matched_random",
            "B1_door", "B2_wall", "B3_isolation", "B4_articulation"]


def read_rows(run, split=SPLIT):
    # keyed by (damage, extent): A1 and A2 appear once per damage extent, so a
    # dictionary keyed by damage alone would keep only the last row
    path = os.path.join(RUNS, run, f"eval_{split}.csv")
    if not os.path.exists(path):
        return {}
    with open(path) as f:
        return {(r["damage"], r["extent"]): r for r in csv.DictReader(f)}


per_run = [x for x in (read_rows(r) for r in M2) if x]
keys = sorted({k for d in per_run for k in d},
              key=lambda k: (ALL_COND.index(k[0]) if k[0] in ALL_COND else 99, float(k[1])))

print(f"{'condition':<20}{'extent':>9}{'topological':>13}{'rooms':>8}"
      f"{'RSR':>16}{'tile accuracy':>18}")
print("-" * 84)
for key in keys:
    entries = [d[key] for d in per_run if key in d]
    rsr_v = [float(e["rsr_mean"]) for e in entries]
    acc_v = [float(e["acc_mean"]) for e in entries]
    e0 = entries[0]
    print(f"{e0['damage']:<20}{e0['extent']:>9}{e0['topological']:>13}{e0['n_rooms']:>8}"
          f"{np.mean(rsr_v):>9.3f}+-{np.std(rsr_v):<6.3f}"
          f"{np.mean(acc_v):>11.3f}+-{np.std(acc_v):<6.3f}")

B2 is marked as non-topological. Damage to impassable structure cannot break
connectivity by construction: if the model leaves the destroyed cells unfilled they
decode to void, which is not walkable, exactly like the wall that was there. That
condition is read on tile accuracy rather than on the topological criterion.

## Qualitative examples

What the failure looks like. Requires the checkpoints, not only the CSV files; the cell
is skipped when they are not available.

The two rows use the same number of destroyed cells. In the first the neighbourhood of
the damage is floor, so the locally plausible completion is also the correct one. In the
second the neighbourhood is wall, the locally plausible completion is wall, and the model
produces it.

In [ ]:
CKPT = os.path.join(RUNS, "m2_seed1", "last.pt")

if not os.path.exists(CKPT):
    print(f"no checkpoint at {CKPT}; skipping the qualitative figure")
else:
    import torch
    from omegaconf import OmegaConf

    from src.damage.targeted import TARGETED, kill_cells
    from src.models.encoding import decode, to_nca_state
    from src.models.factory import build_model
    from src.metrics.connectivity import preserves_topology
    from src.viz import render_room

    saved = OmegaConf.load(os.path.join(RUNS, "m2_seed1", ".hydra", "config.yaml"))
    model = build_model(saved.model)
    model.load_state_dict(torch.load(CKPT, map_location="cpu")["nca"])
    model.eval()

    rooms = np.load("../data/processed/rooms.npz", allow_pickle=True)["rooms"]
    from src.data.splits import train_val_test_split
    _, _, test = train_val_test_split(rooms, saved.data.val_fraction,
                                      saved.data.test_fraction, seed=saved.seed)

    def repair(room, selector, seed=1, steps=96, **kw):
        rng = np.random.default_rng(seed)
        state = to_nca_state(torch.as_tensor(room).unsqueeze(0), saved.model.hidden_channels)
        mask = selector(room, rng, **kw)
        state, _ = kill_cells(state, mask)
        damaged = decode(state)[0].numpy()
        with torch.no_grad():
            for _ in range(steps):
                state = model(state)
        return damaged, decode(state)[0].numpy(), mask

    room = test[0]
    fig, axes = plt.subplots(2, 3, figsize=(6.5, 3.4))
    for row, (name, selector, kw) in enumerate(
            [("B3, floor next to a door", TARGETED["B3_isolation"], {}),
             ("B1, door cells", TARGETED["B1_door"], {"n_doors": 1})]):
        damaged, repaired, mask = repair(room, selector, **kw)
        ok = preserves_topology(room, repaired)
        render_room(room, axes[row][0], "pristine")
        render_room(damaged, axes[row][1], f"{name}\n({int(mask.sum())} cells)", highlight=mask)
        render_room(repaired, axes[row][2], f"repaired, topology {'kept' if ok else 'lost'}")
    plt.tight_layout()
    fig.savefig(f"{FIGDIR}/qualitative.png", bbox_inches="tight")